# FQ-SSIUU Mother Notebook v2 (Local Windows)

**Local execution version** — designed to run on Windows with RTX 3060 12GB,
replacing the Kaggle container environment.

## Quick Start
1. Set `HF_TOKEN` in your system environment (see README)
2. Open this notebook in Jupyter (File → Change Kernel → `.venv/Scripts/python.exe`)
3. Run all cells sequentially
4. Outputs saved to `./runs/<run-id>/`

**Prerequisites:** Python 3.11+, CUDA-capable GPU (12GB VRAM minimum),
virtual environment with dependencies installed.

---
**Author:** Rohan Ahmed (23BSAI008), Muhammad Arslan (23BSAI020)
**Supervisor:** Dr. Sania Bhatti
**Project:** FQ-SSIUU — Verifiable Machine Unlearning Against Quantisation Recovery Attacks

In [2]:
import os
import sys
from pathlib import Path

import torch

# ── 1. Project root ──
ROOT = Path.cwd()

# ── 2. Local model configuration ──
BASE_MODEL_ID = r"C:\Model\Llama-3.2-1B-Instruct"

# Optional: Read the token if one is configured.
HF_TOKEN = os.environ.get("HF_TOKEN")

# Verify that the local model directory exists.
MODEL_PATH = Path(BASE_MODEL_ID)

if not MODEL_PATH.is_dir():
    raise FileNotFoundError(
        f"Model directory not found: {MODEL_PATH}"
    )

# Verify that the directory contains a Hugging Face model config.
if not (MODEL_PATH / "config.json").exists():
    raise FileNotFoundError(
        f"config.json not found in {MODEL_PATH}. "
        "Check whether the model files are stored in this directory."
    )

# ── 3. Output paths ──
RUNS_DIR = ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ── 4. Integration-test recipe ──
RECIPE_PATH = ROOT / "configs" / "runs" / "integration_v1.json"

if not RECIPE_PATH.exists():
    raise FileNotFoundError(
        f"Recipe not found: {RECIPE_PATH}"
    )

# ── 5. Display configuration ──
print(f"Project root: {ROOT}")
print(f"Model path:   {MODEL_PATH}")
print(f"Recipe:       {RECIPE_PATH}")
print(f"Runs dir:     {RUNS_DIR}")
print(f"HF_TOKEN:     {'Set' if HF_TOKEN else 'Not set'}")
print(f"Python:       {sys.version.split()[0]}")
print(
    f"PyTorch:      {torch.__version__}"
)
print(
    f"CUDA:         {'Available' if torch.cuda.is_available() else 'Unavailable'}"
)

if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory:   "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB"
    )

Project root: c:\MY-FYP\fq-ssiuu
Model path:   C:\Model\Llama-3.2-1B-Instruct
Recipe:       c:\MY-FYP\fq-ssiuu\configs\runs\integration_v1.json
Runs dir:     c:\MY-FYP\fq-ssiuu\runs
HF_TOKEN:     Not set
Python:       3.13.14
PyTorch:      2.14.1+cu132
CUDA:         Available
GPU:          NVIDIA GeForce RTX 3060
GPU memory:   12.00 GiB


In [3]:
# ── Imports & Setup ──
import hashlib
import importlib.util
import json
import shutil
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import torch
import numpy as np

# Verify critical imports work
for pkg in ["torch", "transformers", "peft", "datasets", "bitsandbytes"]:
    try:
        __import__(pkg)
        print(f"  OK  {pkg}")
    except ImportError as e:
        print(f"  FAIL {pkg}: {e}")

# GPU check
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)} GB")

# Working directories
run_uuid = uuid.uuid4().hex[:12]
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + run_uuid
checkout_dir = ROOT / "src-checkout" / run_id
provenance_path = RUNS_DIR / f"bootstrap-{run_id}.json"
output_dir = RUNS_DIR / run_id
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nRun ID: {run_id}")
print(f"Checkout: {checkout_dir}")
print(f"Output: {output_dir}")

  OK  torch


c:\VENV\fyp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  OK  transformers


W1024 04:01:03.891000 22340 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


  OK  peft
  OK  datasets
  OK  bitsandbytes

GPU: NVIDIA GeForce RTX 3060
VRAM: 12.0 GB

Run ID: 20261024-110113-d6f87746dc85
Checkout: c:\MY-FYP\fq-ssiuu\src-checkout\20261024-110113-d6f87746dc85
Output: c:\MY-FYP\fq-ssiuu\runs\20261024-110113-d6f87746dc85


In [4]:
# ── Source Manifest & Checkout ──
import json

try:
    # 1. Load source manifest
    manifest_path = ROOT / "SOURCE_MANIFEST.json"
    if not manifest_path.exists():
        # Fallback: try to generate one from git
        print("  -> No SOURCE_MANIFEST.json found. Attempting to generate from git...")
        git_info = {
            "source_origin_sha": subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
            ).strip()[:12],
            "source_tree_sha": subprocess.check_output(
                ["git", "rev-parse", "HEAD^{tree}"], cwd=ROOT, text=True
            ).strip()[:12],
            "created_local": True,
        }
        with manifest_path.open("w") as f:
            json.dump(git_info, f, indent=2)
        print(f"  -> Generated local manifest: {manifest_path}")
    else:
        manifest = json.loads(manifest_path.read_text())
        print(f"  OK  Loaded manifest: {manifest_path}")

    # 2. Verify source tree SHA
    expected_tree = subprocess.check_output(
        ["git", "rev-parse", "HEAD^{tree}"], cwd=ROOT, text=True
    ).strip()[:12]
    print(f"  Source tree SHA: {expected_tree}")

    # 3. Clone repo to checkout dir
    if checkout_dir.exists():
        shutil.rmtree(checkout_dir)
    checkout_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--no-checkout", str(ROOT), str(checkout_dir)],
                   check=True, cwd=ROOT)
    subprocess.run(["git", "checkout", "HEAD", "--", "."], cwd=checkout_dir, check=True)

    # 4. Verify repo is clean
    status = subprocess.run(["git", "status", "--porcelain"], cwd=checkout_dir,
                            capture_output=True, text=True)
    if status.stdout.strip():
        print("  WARNING: Working tree has uncommitted changes. Proceeding...")
    else:
        print("  OK  Source checkout is clean.")

    print("  OK  Source ready for execution.")

except Exception as exc:
    print(f"  FAIL: Source setup failed: {exc}")
    raise

  -> No SOURCE_MANIFEST.json found. Attempting to generate from git...
  -> Generated local manifest: c:\MY-FYP\fq-ssiuu\SOURCE_MANIFEST.json
  Source tree SHA: 44931ced17d4
  OK  Source checkout is clean.
  OK  Source ready for execution.


In [5]:
# ── Model Loading (NF4 4-bit) ──
from fq_ssiuu.models.loader import load_base_model, ModelConfig

print("Loading base model (NF4 4-bit)...")
print(f"Model: {BASE_MODEL_ID}")

# Check VRAM headroom
if torch.cuda.is_available():
    free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"Free VRAM before load: {round(free_mem / 1024**3, 1)} GB")

config = ModelConfig(
    base_model_id=BASE_MODEL_ID,
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    torch_dtype="float16",
    device_map={"": 0},
)

model = load_base_model(config)

if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    res = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory: {alloc:.2f} GB allocated, {res:.2f} GB reserved")

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("OK  Model loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading base model (NF4 4-bit)...
Model: C:\Model\Llama-3.2-1B-Instruct
Free VRAM before load: 12.0 GB


Loading weights: 100%|██████████| 146/146 [00:11<00:00, 12.92it/s]

GPU memory: 0.96 GB allocated, 1.00 GB reserved
Parameters: 749,275,136
OK  Model loaded successfully.


In [6]:
# ── LoRA Wrapping ──
from peft import LoraConfig, get_peft_model

print("Applying LoRA adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in peft_model.parameters())
print(f"LoRA applied. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Check VRAM after LoRA
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    res = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory: {alloc:.2f} GB allocated, {res:.2f} GB reserved")
    if res > 8.0:
        print(f"  WARNING: Only {res:.1f} GB free - may OOM on training")

print("OK  LoRA ready.")

Applying LoRA adapters...
LoRA applied. Trainable: 3,407,872 / 752,683,008 (0.45%)
GPU memory: 0.97 GB allocated, 1.01 GB reserved
OK  LoRA ready.


In [14]:
runner_path = ROOT / "scripts" / "mother_runner.py"
lines = runner_path.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "output_locations" in line or "provenance[" in line or "provenance.get(" in line:
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 74-81 ---
74:     return recipe
75: 
76: 
77: def output_locations(recipe, requested, run_id):
78:     requested = Path(requested).resolve()
79:     sealed = requested.parent / (run_id + "-sealed")
80:     if recipe["kind"] == "adapted_reproduction":
81:         # A native optimizer checkpoint plus FP16 export is ~17 GB. Saving a

--- Lines 111-118 ---
111: def execute(recipe_path, output, provenance, run_id):
112:     recipe_path, output = Path(recipe_path), Path(output)
113:     recipe = load_recipe(recipe_path)
114:     output, sealed_output = output_locations(recipe, output, run_id)
115:     if sealed_output.exists():
116:         raise FileExistsError("Sealed run namespace already exists")
117:     output.mkdir(parents=True, exist_ok=False)
118:     write_new(output / "recipe.json", recipe)

--- Lines 153-160 ---
153:             for name in recipe["required_secrets"]:
154:                 env[name] = get_secret(name)
155:         head = subprocess.check_output(["git", 

In [15]:
import json
import subprocess
import uuid
from datetime import datetime, timezone

# Get the full commit SHA of the current repository
snapshot_commit_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    text=True
).strip()

print("Repository:", ROOT)
print("Snapshot commit:", snapshot_commit_sha)

# Create a fresh run ID so the failed run's output directory is not reused
run_uuid = uuid.uuid4().hex[:12]
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-") + run_uuid

# Recalculate the paths for this new run
provenance_path = RUNS_DIR / f"bootstrap-{run_id}.json"
output_dir = RUNS_DIR / run_id

# Write the provenance record required by mother_runner.py
provenance = {
    "snapshot_commit_sha": snapshot_commit_sha
}

provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8"
)

print("Provenance file:", provenance_path)
print("Provenance exists:", provenance_path.exists())
print("Output path already exists:", output_dir.exists())
print("Provenance contents:")
print(provenance_path.read_text(encoding="utf-8"))

Repository: c:\MY-FYP\fq-ssiuu
Snapshot commit: 9589b4e53a4544c48b9836e137996749dae5a076
Provenance file: c:\MY-FYP\fq-ssiuu\runs\bootstrap-20261024-110738-149f17d4cc7c.json
Provenance exists: True
Output path already exists: False
Provenance contents:
{
  "snapshot_commit_sha": "9589b4e53a4544c48b9836e137996749dae5a076"
}


In [17]:
from pathlib import Path
import json

print("Run output directory:", output_dir)
print("Exists:", output_dir.exists())

if output_dir.exists():
    for path in sorted(output_dir.rglob("*")):
        if path.is_file():
            print(f"\n--- {path.relative_to(output_dir)} ---")

            if path.suffix.lower() == ".json":
                try:
                    data = json.loads(path.read_text(encoding="utf-8"))
                    print(json.dumps(data, indent=2)[:12000])
                except Exception as exc:
                    print(f"Could not parse JSON: {exc}")

Run output directory: c:\MY-FYP\fq-ssiuu\runs\20261024-110738-149f17d4cc7c
Exists: True

--- environment\before_distributions.json ---
{
  "schema_version": "1.1",
  "stage": "before",
  "run_id": "99535044-fe90-4106-86ea-e7cbe753f1a5",
  "git_sha": "9589b4e53a4544c48b9836e137996749dae5a076",
  "source": "importlib.metadata.distributions",
  "packages": {
    "accelerate": "1.15.0",
    "aiohappyeyeballs": "2.7.1",
    "aiohttp": "3.14.5",
    "aiosignal": "1.4.0",
    "annotated-doc": "0.0.5",
    "anyio": "4.15.1",
    "asttokens": "3.0.2",
    "attrs": "26.1.0",
    "bitsandbytes": "0.50.2",
    "captum": "0.9.0",
    "click": "8.5.0",
    "colorama": "0.4.6",
    "comm": "0.2.3",
    "contourpy": "1.4.0",
    "cycler": "0.12.1",
    "datasets": "5.1.0",
    "debugpy": "1.8.22",
    "dill": "0.4.1",
    "dotenv": "0.9.9",
    "executing": "2.3.0",
    "filelock": "3.32.3",
    "fonttools": "4.67.0",
    "fq-ssiuu": "0.1.0.dev0",
    "frozenlist": "1.8.0",
    "fsspec": "2026.7.0",
 

In [18]:
runner_path = ROOT / "scripts" / "mother_runner.py"
lines = runner_path.read_text(encoding="utf-8").splitlines()

keywords = [
    "failure_phase",
    "failure_type",
    "environment",
    "traceback",
    "stderr",
    "except RuntimeError",
]

for i, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")


--- Lines 94-101 ---
94:     try:
95:         with (
96:             (output / f"{name}_stdout.log").open("xb") as stdout,
97:             (output / f"{name}_stderr.log").open("xb") as stderr,
98:         ):
99:             result = subprocess.run(
100:                 argv, cwd=ROOT, env=env, stdout=stdout, stderr=stderr, timeout=timeout, check=False
101:             )

--- Lines 97-104 ---
97:             (output / f"{name}_stderr.log").open("xb") as stderr,
98:         ):
99:             result = subprocess.run(
100:                 argv, cwd=ROOT, env=env, stdout=stdout, stderr=stderr, timeout=timeout, check=False
101:             )
102:         record["returncode"] = result.returncode
103:     except subprocess.TimeoutExpired:
104:         record.update(returncode=124, termination="TIMEOUT")

--- Lines 142-149 ---
142:     )
143:     if recipe["kind"] == "adapted_reproduction":
144:         env.update(DS_BUILD_OPS="0", HF_HOME="/kaggle/tmp/fq-d1-hf-cache")
145:     phase = "envir

In [19]:
from pathlib import Path

log_dir = output_dir / "environment"

print("Run directory:", output_dir)
print("Environment log directory:", log_dir)
print("Environment directory exists:", log_dir.exists())

# List every file produced by the run
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_dir))

# Print the environment setup logs
for name in ("environment_stdout.log", "environment_stderr.log"):
    path = output_dir / name
    if path.exists():
        print(f"\n{'=' * 20} {name} {'=' * 20}")
        print(path.read_text(encoding="utf-8", errors="replace")[-12000:])

# Some versions may store the logs inside the environment directory
if log_dir.exists():
    for path in sorted(log_dir.rglob("*.log")):
        print(f"\n{'=' * 20} {path.relative_to(output_dir)} {'=' * 20}")
        print(path.read_text(encoding="utf-8", errors="replace")[-12000:])

Run directory: c:\MY-FYP\fq-ssiuu\runs\20261024-110738-149f17d4cc7c
Environment log directory: c:\MY-FYP\fq-ssiuu\runs\20261024-110738-149f17d4cc7c\environment
Environment directory exists: True
environment\before_distributions.json
environment\before_freeze.log
environment\before_manifest.json
environment\MANIFEST.json
environment\native_baseline.json
environment\post_attempt_manifest.json
environment\setup_status.json
environment_execution.json
environment_stderr.log
environment_stdout.log
recipe.json
run_result.json
run_started.json
source_provenance.json

==================== environment_stdout.log ====================
=== KAGGLE OVERLAY SETUP (MODE: APPLY) ===


==================== environment_stderr.log ====================
ERROR: Environment before-manifest failed baseline validation! Setup stopped.


==================== environment\before_freeze.log ====================
accelerate==1.15.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiosignal==1.4.0
annotated-doc==0.0.5
anyio==4.1

In [21]:
from pathlib import Path

files_to_inspect = [
    ROOT / "scripts" / "kaggle_setup.py",
    ROOT / "requirements" / "kaggle-overlay.validated.lock",
    ROOT / "configs" / "reproductions" / "NPO" / "d1_train_v1.json",
]

for path in files_to_inspect:
    print(f"\n{'=' * 80}")
    print(f"FILE: {path.relative_to(ROOT)}")
    print(f"{'=' * 80}")

    if path.exists():
        content = path.read_text(encoding="utf-8", errors="replace")
        print(content[:15000])
    else:
        print("FILE NOT FOUND")


FILE: scripts\kaggle_setup.py
#!/usr/bin/env python3
"""
kaggle_setup.py — Orchestration script for Kaggle overlay dependency installation.

Fail-closed implementation:
- Preview is the default mode.
- Requires explicit --apply flag for installation.
- --apply and --preview are mutually exclusive.
- --report-file may only be used with --preview for offline testing.
- Refuses to operate unless CUDA_VISIBLE_DEVICES=0.
- Refuses to overwrite existing output directory.
- Strictly validates baseline structure using validate_baseline before all actions.
- Captures genuine before/after/post-attempt environment manifests using common run_id.
- Validates dry-run pip report and actual installation report against requirements/kaggle-native-baseline.json.
- Installs repository editable with --no-deps only after successful validation.
- Runs post-install verifier (scripts/verify_kaggle_env.py) with common run_id and checks run_id, git_sha, and overall status.
- Generates verifier_execution.json ar

In [20]:
# ── Run the Mother Runner (Integration Recipe) ──
import subprocess
import json

print("Executing mother_runner.py...")
print(f"Recipe: {RECIPE_PATH}")
print(f"Output: {output_dir}")

mother_runner = ROOT / "scripts" / "mother_runner.py"
if not mother_runner.exists():
    print("  FAIL: mother_runner.py not found. Skipping execution.")
    raise FileNotFoundError("mother_runner.py not found")

command = [
    sys.executable, str(mother_runner),
    "--recipe", str(RECIPE_PATH),
    "--output", str(output_dir),
    "--provenance", str(provenance_path),
    "--run-id", run_id,
]

env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
print(f"Running: {' '.join(command[:3])}...")

result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=3600)

print("STDOUT (tail):")
print(result.stdout[-1000:] if result.stdout else "None")
print("STDERR (tail):")
print(result.stderr[-1000:] if result.stderr else "None")
print(f"Return code: {result.returncode}")

if result.returncode != 0:
    raise RuntimeError(f"Recipe failed with code {result.returncode}")

print(f"\nOK  Run complete: {run_id}")
print(f"  Outputs: {output_dir}")

Executing mother_runner.py...
Recipe: c:\MY-FYP\fq-ssiuu\configs\runs\integration_v1.json
Output: c:\MY-FYP\fq-ssiuu\runs\20261024-110738-149f17d4cc7c
Running: c:\VENV\fyp\Scripts\python.exe c:\MY-FYP\fq-ssiuu\scripts\mother_runner.py --recipe...
STDOUT (tail):
None
STDERR (tail):
Traceback (most recent call last):
  File "c:\MY-FYP\fq-ssiuu\scripts\mother_runner.py", line 315, in <module>
    outcome = execute(
        args.recipe, args.output, json.loads(Path(args.provenance).read_text()), args.run_id
    )
  File "c:\MY-FYP\fq-ssiuu\scripts\mother_runner.py", line 116, in execute
    raise FileExistsError("Sealed run namespace already exists")
FileExistsError: Sealed run namespace already exists

Return code: 1


RuntimeError: Recipe failed with code 1

In [ ]:
# ── Save Provenance & Summary ──
import json as json_mod
from datetime import UTC

provenance = {
    "run_id": run_id,
    "source_origin_sha": subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()[:12],
    "git_branch": subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only",
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0,
    "python_version": sys.version,
    "timestamp": datetime.now(UTC).isoformat(),
    "status": "SUCCESS" if result.returncode == 0 else "FAILURE",
}

with provenance_path.open("x") as f:
    json_mod.dump(provenance, f, indent=2)

print(f"OK  Provenance saved: {provenance_path}")
print(f"\n=== SUMMARY ===")
print(f"Run ID:    {run_id}")
print(f"Status:    {provenance['status']}")
print(f"GPU:       {provenance['gpu']}")
print(f"VRAM:      {provenance['vram_gb']} GB")
print(f"Outputs:   {output_dir}")

print("\n=== NEXT STEPS ===")
print("1. Inspect outputs in:  runs/<run-id>")
print("2. Run tests:            .venv/Scripts/python -m pytest tests -v")
print("3. Archive:              python scripts/archive_run.py seal --run-dir runs/<run-id> --output runs/sealed --run-id <run-id>")
print("4. Save to git:          git add -A && git commit -m 'Local execution' && git push")
print("5. View report:          .venv/Scripts/python -m markdown <output>.md")

In [ ]:
# ── Cleanup ──
import shutil

if checkout_dir.exists():
    shutil.rmtree(checkout_dir)
    print(f"OK  Cleanup: Removed {checkout_dir}")

print("\nNotebook execution complete.")
print("See 'runs/' for all run artifacts.")